# Notebook 1 - kinematische analyse

Deze notebook is een compacte, inhoudelijke versie van de volledige kinematische analyse van het paraplu-mechanisme. De volledige notebook bevat veel achtergrond en afleiding; deze versie behoudt de noodzakelijke theorie, de gebruikte aannames, de berekeningen en de figuren die nodig zijn om de werking van het mechanisme snel te begrijpen. Alle berekeningen vertrekken van dezelfde parameters en dezelfde sluitingsvergelijkingen als in de originele Notebook 1.

De analyse beschouwt het mechanisme als een vlak stangenmechanisme met starre links en ideale gewrichten. Alle bewegingen liggen dus in ??n vlak. De topologie wordt bepaald door de verbindingen tussen de links, niet door toevallige kruisingen in de schets. Een kruising van twee staven is alleen een gewricht wanneer dat expliciet in de modellering aanwezig is. Link 1 is de vaste grondlink, link 2 is de schuiver en de overige links vormen de beweeglijke stangen van het paraplu-mechanisme.

De notebook is zo opgebouwd dat een wijziging in de parametercel bovenaan doorwerkt naar alle latere resultaten. Na `Run All` zijn de positie, snelheid, versnelling, animatie, conditionering en overdracht naar Notebook 2 opnieuw consistent met dezelfde geometrie en hetzelfde schuivertraject.

## Onderwerpenoverzicht

| Onderwerp | Sectie | Inhoudelijke rol |
|---|---|---|
| Mechanisme en geometrie | `Parameters` | Definieert de linklengtes, het schuivertraject en de begintoestand. |
| Mobiliteit en invoer | `Parameters` | Legt vast dat het mechanisme ??n vrijheidsgraad heeft en door `s(t)` wordt aangedreven. |
| Positie van het mechanisme | `Positiecontrole` | Lost de sluitingsvergelijkingen op voor alle tijdstappen. |
| Outputpunt K | `Positiecontrole`, `Snelheidsanalyse`, `Versnellingsanalyse` | Beschrijft de verplaatsing, snelheid en versnelling van het uiteinde van de rib. |
| Geometrische controle | `Animatie` | Controleert visueel of de berekende configuraties fysisch samenhangen. |
| Snelheidsstelsel | `Snelheidsanalyse` | Gebruikt de afgeleide sluitingsvergelijkingen en matrix `A`. |
| Dynamisch gevoelige zones | `Versnellingsanalyse` | Bepaalt waar inerti?le effecten groter kunnen worden. |
| Singulariteiten en dode punten | `Conditionering en singulariteiten` | Interpreteert de condition number van de snelheidsmatrix. |
| Koppeling met dynamica | `Opslag voor Notebook 2` | Schrijft de kinematische grootheden weg naar `.npz`. |

De nadruk ligt op kinematica: posities, snelheden, versnellingen en numerieke gevoeligheid. Krachten, onbalans en motorbelasting worden hier niet opnieuw afgeleid, maar deze notebook levert wel de grootheden die daarvoor nodig zijn. De scheiding tussen kinematica en dynamica voorkomt dat dezelfde berekening op verschillende plaatsen met mogelijk afwijkende parameters wordt uitgevoerd.


## Topologie van het paraplu-mechanisme

Het mechanisme bestaat uit 8 links (inclusief de vaste grond) en
10 gewrichten (B, C, D, E, F, G, H, I, J, K).

| Link | Van → Naar | Type | Rol |
|------|-----------|------|-----|
| 1 | vaste mast | grond | referentie |
| 2 | (schuiver langs mast) | prismatisch | invoer s(t) |
| 3 | B → D → E | ternair | koppelt schuiver aan link 4 |
| 4 | C → E → H | ternair | ruggengraat, vast scharnier in C |
| 5 | D → F → G | ternair | eerste radiale uitbreiding |
| 6 | F → I | binair | koppelstang |
| 7 | G → H → J | ternair | synchronisatiestang |
| 8 | I → J → K | ternair | buitenste rib — K is outputpunt |

C is het enige vaste rotatiepunt. Punt K is de tip van het parapludoek.

## Parameters

De parametercel bevat alle grootheden die de kinematische berekening rechtstreeks bepalen. De mastlengte `L1` legt de vaste referentie vast. De parameters `r3a`, `r3b`, ..., `r8b` zijn de geometrische lengtes van de stangen en deelstukken van samengestelde links. Suffixen zoals `3a` en `3b` betekenen niet dat er twee aparte lichamen zijn; ze geven twee segmenten van dezelfde starre link aan.

Het mechanisme wordt gemodelleerd met link 1 als grond. De oorsprong van het assenstelsel ligt in punt `C`, met de `x`-as horizontaal en de `y`-as verticaal. De schuiver beweegt langs de vaste mast. Zijn positie wordt beschreven door de invoercoordinaat `s(t)`. Grote `s` komt overeen met een meer gesloten toestand; kleine `s` met een meer open toestand.

Het bewegingsprofiel is afgestemd op een positiegestuurde lineaire actuator. De standaardkeuze is `condition_scurve`: eerst wordt over het schuiverbereik een quasi-statische condition-map gemaakt, daarna beweegt de schuiver trager in slecht geconditioneerde zones en sneller waar de overbrenging gunstiger is. De eindpunten blijven glad: de snelheid en versnelling zijn nul in gesloten en open stand.

De opties `smooth_345` en `smooth_4567` blijven beschikbaar als eenvoudige globale referentietrajecten. `smooth_4567` is zachter aan start en einde dan `smooth_345`, terwijl `condition_scurve` daar bovenop de snelheid positioneel verlaagt bij hoge `cond(A)`. De parameters `t_move_desired`, `auto_extend_t_move`, `actuator_v_limit`, `actuator_a_limit`, `condition_slow_gain` en `condition_min_speed_factor` maken het traject snel aanpasbaar zonder de kinematische analyse zelf te wijzigen.

Voor tussenstanden kan `motion_direction = "custom"` gebruikt worden met eigen waarden voor `s_start_custom` en `s_end_custom`. De mobiliteit volgt uit de vlakke Gruebler/Kutzbach-formule `M = 3(n - 1) - 2 f1 - f2`; het mechanisme heeft een vrijheidsgraad, waardoor `s(t)` als enige onafhankelijke invoer volstaat.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from scipy.interpolate import PchipInterpolator
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

%matplotlib widget

# ==========================================================
# Parameters van het paraplu-mechanisme
# Deze cel bevat de gegevens die snel aangepast kunnen worden.
# ==========================================================

# Mastlengte
L1 = 2.5  # [m]

# Geometrische parameters van de links [m]
r3a = 0.507375
r3b = 0.428535
r4a = 0.941041
r4b = 0.655989
r5a = 0.251430
r5b = 0.404558
r6  = 0.680427
r7a = 0.428535
r7b = 0.251892
r8a = 0.404558
r8b = 0.499793

# Invoercoordinaat s:
# s wordt naar beneden gemeten vanaf punt C.
# grote s = meer gesloten toestand; kleine s = meer open toestand.
s_open = 0.125000      # minimale schuiverpositie [m] = open stand
s_closed = 1.875000    # maximale schuiverpositie [m] = gesloten stand

# Bewegingsopdracht voor de lineaire actuator
# "open":   gesloten -> open -> vasthouden
# "close":  open -> gesloten -> vasthouden
# "custom": gebruik s_start_custom en s_end_custom, bv. voor tussenstanden
motion_direction = "open"
s_start_custom = s_closed
s_end_custom = s_open

# Trajectparameters
motion_profile = "condition_scurve"  # "condition_scurve", "smooth_4567" of "smooth_345"
t_begin = 0.0             # begintijd [s]
t_move_desired = 14.0     # gewenste tijd om van startstand naar eindstand te bewegen [s]
t_hold = 4.0              # extra stilstand in de eindstand [s]
Ts = 0.05                 # tijdstap [s]
auto_extend_t_move = True # verleng t_move automatisch als actuatorlimieten overschreden worden

# Richtwaarden voor een realistische actuatorbeweging.
# Deze limieten wijzigen alleen t_move indien auto_extend_t_move = True.
actuator_v_limit = 0.25   # [m/s] maximale gewenste schuiversnelheid
actuator_a_limit = 0.10   # [m/s^2] maximale gewenste schuiverversnelling

# Parameters voor het condition-aware traject.
# Een hogere gain vertraagt sterker bij hoge cond(A).
condition_slow_gain = 1.25
condition_min_speed_factor = 0.25
condition_map_points = 301
trajectory_shape_points = 3001

# Initiele gok voor de positie-oplosser.
# Deze beginwaarden bepalen mee welke configuratietak gevolgd wordt.
theta3_init = 1.432805
theta4_init = -1.435562
theta5_init = -1.435562
theta6_init = 1.432805
theta7_init = 1.432805
theta8_init = -1.435562
initial_theta_guess = np.array([theta3_init, theta4_init, theta5_init, theta6_init, theta7_init, theta8_init])

if t_move_desired <= 0 or t_hold < 0:
    raise ValueError("t_move_desired moet positief zijn en t_hold mag niet negatief zijn.")
if not (0.0 < condition_min_speed_factor <= 1.0):
    raise ValueError("condition_min_speed_factor moet in het interval (0, 1] liggen.")

if motion_direction == "open":
    s_start = s_closed
    s_end = s_open
elif motion_direction == "close":
    s_start = s_open
    s_end = s_closed
elif motion_direction == "custom":
    s_start = s_start_custom
    s_end = s_end_custom
else:
    raise ValueError("motion_direction moet 'open', 'close' of 'custom' zijn.")

s_min, s_max = sorted([s_open, s_closed])
if not (s_min <= s_start <= s_max and s_min <= s_end <= s_max):
    raise ValueError("s_start en s_end moeten binnen het bereik [s_open, s_closed] liggen.")
if np.isclose(s_start, s_end):
    raise ValueError("s_start en s_end mogen niet gelijk zijn voor een bewegingstraject.")

delta_s = s_end - s_start


def static_loop_closure_eqs(theta, s_k):
    theta3, theta4, theta5, theta6, theta7, theta8 = theta

    F1 = (r3a + r3b) * np.cos(theta3) - r4a * np.cos(theta4)
    F2 = -s_k + (r3a + r3b) * np.sin(theta3) - r4a * np.sin(theta4)

    F3 = -r3b * np.cos(theta3) + (r5a + r5b) * np.cos(theta5) \
         + r7a * np.cos(theta7) - r4b * np.cos(theta4)
    F4 = -r3b * np.sin(theta3) + (r5a + r5b) * np.sin(theta5) \
         + r7a * np.sin(theta7) - r4b * np.sin(theta4)

    F5 = -r5b * np.cos(theta5) + r6 * np.cos(theta6) \
         + r8a * np.cos(theta8) - (r7a + r7b) * np.cos(theta7)
    F6 = -r5b * np.sin(theta5) + r6 * np.sin(theta6) \
         + r8a * np.sin(theta8) - (r7a + r7b) * np.sin(theta7)

    return np.array([F1, F2, F3, F4, F5, F6])


def static_velocity_matrix(theta):
    theta3, theta4, theta5, theta6, theta7, theta8 = theta
    return np.array([
        [-(r3a + r3b) * np.sin(theta3),  r4a * np.sin(theta4), 0, 0, 0, 0],
        [ (r3a + r3b) * np.cos(theta3), -r4a * np.cos(theta4), 0, 0, 0, 0],

        [ r3b * np.sin(theta3),          r4b * np.sin(theta4), -(r5a + r5b) * np.sin(theta5), 0, -r7a * np.sin(theta7), 0],
        [-r3b * np.cos(theta3),         -r4b * np.cos(theta4),  (r5a + r5b) * np.cos(theta5), 0,  r7a * np.cos(theta7), 0],

        [0, 0,  r5b * np.sin(theta5), -r6 * np.sin(theta6),  (r7a + r7b) * np.sin(theta7), -r8a * np.sin(theta8)],
        [0, 0, -r5b * np.cos(theta5),  r6 * np.cos(theta6), -(r7a + r7b) * np.cos(theta7),  r8a * np.cos(theta8)]
    ])


def smooth_shape(u, profile):
    if profile == "smooth_345":
        h = 10*u**3 - 15*u**4 + 6*u**5
        dh = 30*u**2 - 60*u**3 + 30*u**4
        ddh = 60*u - 180*u**2 + 120*u**3
    elif profile == "smooth_4567":
        h = 35*u**4 - 84*u**5 + 70*u**6 - 20*u**7
        dh = 140*u**3 - 420*u**4 + 420*u**5 - 140*u**6
        ddh = 420*u**2 - 1680*u**3 + 2100*u**4 - 840*u**5
    else:
        raise ValueError("profile moet 'smooth_345' of 'smooth_4567' zijn.")
    return h, dh, ddh


def cumulative_trapezoid(y, x):
    out = np.zeros_like(x, dtype=float)
    out[1:] = np.cumsum(0.5 * (y[1:] + y[:-1]) * np.diff(x))
    return out


def interp_by_s(s_values, s_grid, y_grid):
    order = np.argsort(s_grid)
    return np.interp(s_values, s_grid[order], y_grid[order])


def compute_condition_map(s_start_local, s_end_local):
    s_map = np.linspace(s_start_local, s_end_local, condition_map_points)
    theta_guess = initial_theta_guess.copy()
    cond_map = np.zeros_like(s_map)
    residual_map = np.zeros_like(s_map)
    theta_map = np.zeros((len(s_map), 6))
    failures = 0

    for k, s_k in enumerate(s_map):
        sol, _, ier, _ = fsolve(
            lambda theta: static_loop_closure_eqs(theta, s_k),
            theta_guess,
            full_output=True
        )
        if ier != 1:
            failures += 1

        A_k = static_velocity_matrix(sol)
        cond_map[k] = np.linalg.cond(A_k)
        residual_map[k] = np.linalg.norm(static_loop_closure_eqs(sol, s_k))
        theta_map[k] = sol
        theta_guess = sol

    return s_map, cond_map, residual_map, theta_map, failures


def smoothstep(q):
    q = np.clip(q, 0.0, 1.0)
    return q**3 * (10.0 - 15.0*q + 6.0*q**2)


def speed_factor_from_condition(cond_values):
    finite = cond_values[np.isfinite(cond_values)]
    cond_min = np.min(finite)
    cond_max = np.max(finite)
    if np.isclose(cond_min, cond_max):
        cond_score = np.zeros_like(cond_values)
    else:
        # Log-schaal maakt het verschil tussen normale en matig slechte standen geleidelijker.
        cond_score = (np.log(cond_values) - np.log(cond_min)) / (np.log(cond_max) - np.log(cond_min))
    cond_score = smoothstep(cond_score)
    return 1.0 - (1.0 - condition_min_speed_factor) * cond_score**condition_slow_gain


def make_speed_factor_interpolator(s_values, factor_values):
    order = np.argsort(s_values)
    s_sorted = np.asarray(s_values)[order]
    factor_sorted = np.asarray(factor_values)[order]
    return PchipInterpolator(s_sorted, factor_sorted, extrapolate=True)


s_condition_map, cond_condition_map, residual_condition_map, theta_condition_map, condition_map_failures = compute_condition_map(s_start, s_end)
speed_factor_condition_map = speed_factor_from_condition(cond_condition_map)
speed_factor_interpolator = make_speed_factor_interpolator(s_condition_map, speed_factor_condition_map)
speed_factor_derivative = speed_factor_interpolator.derivative()


def build_actuator_trajectory(t_move_local):
    t_local = np.arange(t_begin, t_begin + t_move_local + t_hold + Ts, Ts)
    tau = np.clip((t_local - t_begin) / t_move_local, 0.0, 1.0)

    if motion_profile == "condition_scurve":
        x_grid = np.linspace(0.0, 1.0, trajectory_shape_points)
        h_grid, _, _ = smooth_shape(x_grid, "smooth_4567")
        s_grid = s_start + delta_s * h_grid
        factor_grid = np.clip(speed_factor_interpolator(s_grid), condition_min_speed_factor, 1.0)

        # Reparameteriseer de smooth_4567-kromme: lagere speed factor betekent meer tijd in die zone.
        scaled_time_density = 1.0 / factor_grid
        scaled_time = cumulative_trapezoid(scaled_time_density, x_grid)
        scaled_time_total = scaled_time[-1]
        scaled_time_norm = scaled_time / scaled_time_total

        x_tau = np.interp(tau, scaled_time_norm, x_grid)
        h, dh_dx, ddh_dx2 = smooth_shape(x_tau, "smooth_4567")
        s_tau = s_start + delta_s * h
        factor_tau = np.clip(speed_factor_interpolator(s_tau), condition_min_speed_factor, 1.0)
        dfactor_ds_tau = speed_factor_derivative(s_tau)
        factor_raw_tau = speed_factor_interpolator(s_tau)
        clipped_factor = (factor_raw_tau <= condition_min_speed_factor) | (factor_raw_tau >= 1.0)
        dfactor_ds_tau = np.where(clipped_factor, 0.0, dfactor_ds_tau)
        dfactor_dx_tau = dfactor_ds_tau * delta_s * dh_dx

        dx_dt = scaled_time_total * factor_tau / t_move_local
        ddx_dt2 = scaled_time_total**2 * factor_tau * dfactor_dx_tau / t_move_local**2

        s_local = s_tau
        ds_local = delta_s * dh_dx * dx_dt
        dds_local = delta_s * (ddh_dx2 * dx_dt**2 + dh_dx * ddx_dt2)
        factor_path = factor_tau
    elif motion_profile in ("smooth_345", "smooth_4567"):
        h, dh, ddh = smooth_shape(tau, motion_profile)
        s_local = s_start + delta_s * h
        ds_local = delta_s * dh / t_move_local
        dds_local = delta_s * ddh / t_move_local**2
        factor_path = np.ones_like(t_local)
    else:
        raise ValueError("motion_profile moet 'condition_scurve', 'smooth_4567' of 'smooth_345' zijn.")

    hold_mask_local = t_local >= (t_begin + t_move_local)
    s_local[hold_mask_local] = s_end
    ds_local[hold_mask_local] = 0.0
    dds_local[hold_mask_local] = 0.0
    factor_path[hold_mask_local] = factor_path[~hold_mask_local][-1] if np.any(~hold_mask_local) else 1.0

    return t_local, s_local, ds_local, dds_local, factor_path


def required_time_scale(ds_local, dds_local):
    scale_candidates = [1.0]
    if actuator_v_limit is not None and actuator_v_limit > 0:
        scale_candidates.append(np.max(np.abs(ds_local)) / actuator_v_limit)
    if actuator_a_limit is not None and actuator_a_limit > 0:
        scale_candidates.append(np.sqrt(np.max(np.abs(dds_local)) / actuator_a_limit))
    return max(scale_candidates)


t_move = t_move_desired
for _ in range(4):
    t, s, ds, dds, trajectory_speed_factor = build_actuator_trajectory(t_move)
    scale = required_time_scale(ds, dds)
    if not auto_extend_t_move or scale <= 1.000001:
        break
    # Kleine marge om discretisatie door Ts niet net boven de limiet te laten eindigen.
    t_move *= scale * 1.002

t, s, ds, dds, trajectory_speed_factor = build_actuator_trajectory(t_move)
t_end = t[-1]
hold_mask = t >= (t_begin + t_move)

max_slider_speed = np.max(np.abs(ds))
max_slider_acc = np.max(np.abs(dds))
i_cond_max = int(np.argmax(cond_condition_map))
i_speed_factor_min = int(np.argmin(speed_factor_condition_map))

print("Actuatortraject:")
print(f"richting/profiel              : {motion_direction} / {motion_profile}")
print(f"s_start -> s_end             : {s_start:.4f} m -> {s_end:.4f} m")
print(f"slag                          : {abs(delta_s):.4f} m")
print(f"t_move gewenst / effectief    : {t_move_desired:.2f} s / {t_move:.2f} s")
print(f"t_hold / t_end                : {t_hold:.2f} s / {t_end:.2f} s")
print(f"max |ds|                     : {max_slider_speed:.4f} m/s")
print(f"max |dds|                    : {max_slider_acc:.4f} m/s^2")
if auto_extend_t_move and t_move > t_move_desired * 1.0005:
    print("auto_extend_t_move heeft de beweging verlengd om binnen de actuatorlimieten te blijven.")
elif actuator_v_limit is not None and max_slider_speed > actuator_v_limit:
    print(f"Waarschuwing: max |ds| is groter dan actuator_v_limit = {actuator_v_limit:.4f} m/s")
elif actuator_a_limit is not None and max_slider_acc > actuator_a_limit:
    print(f"Waarschuwing: max |dds| is groter dan actuator_a_limit = {actuator_a_limit:.4f} m/s^2")
print()
print("Condition-aware planning:")
print(f"cond(A) min / gem / max       : {np.min(cond_condition_map):.3e} / {np.mean(cond_condition_map):.3e} / {np.max(cond_condition_map):.3e}")
print(f"hoogste cond(A)               : bij s = {s_condition_map[i_cond_max]:.4f} m")
print(f"laagste snelheidsfactor       : {speed_factor_condition_map[i_speed_factor_min]:.3f} bij s = {s_condition_map[i_speed_factor_min]:.4f} m")
print(f"max residual condition-map    : {np.max(residual_condition_map):.3e}")
if condition_map_failures:
    print(f"Waarschuwing: condition-map had {condition_map_failures} fsolve-stappen zonder perfecte convergentie.")

# Controlefiguur voor het opgelegde actuatortraject.
fig_traj, ax_traj = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
fig_traj.suptitle("Controle van het actuatortraject")
ax_traj[0, 0].plot(t, s)
ax_traj[0, 0].set_title("Schuiverpositie")
ax_traj[0, 0].set_xlabel("t [s]")
ax_traj[0, 0].set_ylabel("s [m]")
ax_traj[0, 0].grid(True)

ax_traj[0, 1].plot(t, ds)
ax_traj[0, 1].axhline(0.0, color="black", linewidth=0.8)
ax_traj[0, 1].set_title("Schuiversnelheid")
ax_traj[0, 1].set_xlabel("t [s]")
ax_traj[0, 1].set_ylabel("ds [m/s]")
ax_traj[0, 1].grid(True)

ax_traj[1, 0].plot(t, dds)
ax_traj[1, 0].axhline(0.0, color="black", linewidth=0.8)
ax_traj[1, 0].set_title("Schuiverversnelling")
ax_traj[1, 0].set_xlabel("t [s]")
ax_traj[1, 0].set_ylabel("dds [m/s^2]")
ax_traj[1, 0].grid(True)

ax_cond = ax_traj[1, 1]
ax_factor = ax_cond.twinx()
ax_cond.plot(s_condition_map, cond_condition_map, label="cond(A)", color="tab:blue")
ax_factor.plot(s_condition_map, speed_factor_condition_map, label="snelheidsfactor", color="tab:orange")
ax_cond.set_title("Conditionering en snelheidsfactor")
ax_cond.set_xlabel("s [m]")
ax_cond.set_ylabel("cond(A)", color="tab:blue")
ax_factor.set_ylabel("factor [-]", color="tab:orange")
ax_cond.grid(True)
lines_1, labels_1 = ax_cond.get_legend_handles_labels()
lines_2, labels_2 = ax_factor.get_legend_handles_labels()
ax_cond.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")
plt.show()

# Backwards-compatible namen voor latere cellen en figuren
s_lower = s_open
s_upper = s_closed


## Positiecontrole

De positie-analyse vertrekt van sluitingsvergelijkingen. Voor elke onafhankelijke kinematische lus moet de vectorsom van alle opeenvolgende verplaatsingen nul zijn. In dit mechanisme zijn drie onafhankelijke lussen nodig. Elke vectori?le lus levert twee scalaire vergelijkingen, zodat er in totaal zes vergelijkingen ontstaan. Dat komt overeen met de zes onbekende ori?ntatiehoeken `theta3`, `theta4`, `theta5`, `theta6`, `theta7` en `theta8`.

De functie `loop_closure_eqs` stelt deze vergelijkingen op voor ??n waarde van de schuiverpositie `s_k`. De functie `kinematics_umbrella` doorloopt daarna alle tijdstappen. Voor elke tijdstap wordt `fsolve` gebruikt om de hoeken te vinden die de sluitingsvergelijkingen voldoen. De oplossing van de vorige tijdstap wordt telkens gebruikt als startpunt voor de volgende tijdstap. Daardoor blijft de beweging continu en wordt vermeden dat de solver naar een andere configuratietak springt.

Na het oplossen worden de relevante punten van het mechanisme opnieuw opgebouwd uit de berekende hoeken en linklengtes. Daaruit volgen onder andere de co?rdinaten van punt `K`. De eerste figuur controleert de evolutie van de hoeken, de positie van `K`, het traject van `K` en de numerieke sluitingsfout. De sluitingsfout is belangrijk omdat een kleine fout bevestigt dat de berekende configuratie de geometrische beperkingen van het mechanisme respecteert.

De positiecontrole is de basis voor alle latere stappen. Snelheden en versnellingen zijn alleen zinvol wanneer de onderliggende positie-oplossing consistent is.

De lussen leggen telkens dezelfde fysieke eis op: wanneer men via verbonden links rond een gesloten keten loopt, moet men terug op hetzelfde punt uitkomen. De sluitingsfout `residual_pos` vat samen hoe goed dit numeriek lukt. Een fout in de orde van machineprecisie betekent dat resterende afwijkingen vooral door numerieke afronding komen, niet door een open of inconsistent mechanisme.


In [ ]:
# Hulpfuncties en kinematische analyse van het paraplu-mechanisme

def rotate_vector(z, theta):
    rotation_matrix = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])
    return np.dot(rotation_matrix, z)


def loop_closure_eqs(theta_init, s_k,
                     r3a, r3b, r4a, r4b, r5a, r5b, r6, r7a, r7b, r8a):
    theta3 = theta_init[0]
    theta4 = theta_init[1]
    theta5 = theta_init[2]
    theta6 = theta_init[3]
    theta7 = theta_init[4]
    theta8 = theta_init[5]

    # Lus 1
    F1 = (r3a + r3b) * np.cos(theta3) - r4a * np.cos(theta4)
    F2 = -s_k + (r3a + r3b) * np.sin(theta3) - r4a * np.sin(theta4)

    # Lus 2
    F3 = -r3b * np.cos(theta3) + (r5a + r5b) * np.cos(theta5) \
         + r7a * np.cos(theta7) - r4b * np.cos(theta4)
    F4 = -r3b * np.sin(theta3) + (r5a + r5b) * np.sin(theta5) \
         + r7a * np.sin(theta7) - r4b * np.sin(theta4)

    # Lus 3
    F5 = -r5b * np.cos(theta5) + r6 * np.cos(theta6) \
         + r8a * np.cos(theta8) - (r7a + r7b) * np.cos(theta7)
    F6 = -r5b * np.sin(theta5) + r6 * np.sin(theta6) \
         + r8a * np.sin(theta8) - (r7a + r7b) * np.sin(theta7)

    return [F1, F2, F3, F4, F5, F6]


def kinematics_umbrella(r3a, r3b, r4a, r4b, r5a, r5b, r6, r7a, r7b, r8a,
                        s, ds, dds,
                        theta3_init, theta4_init, theta5_init, theta6_init, theta7_init, theta8_init,
                        t):

    optim_options = {"full_output": True}

    dt = t[1] - t[0]
    for k, s_k in enumerate(s):

        # Positie-analyse
        x, _, ier, message = fsolve(
            lambda x: loop_closure_eqs(
                x, s_k,
                r3a, r3b, r4a, r4b, r5a, r5b, r6, r7a, r7b, r8a
            ),
            [theta3_init, theta4_init, theta5_init, theta6_init, theta7_init, theta8_init],
            **optim_options
        )

        if ier != 1:
            print(f"Waarschuwing: fsolve convergeerde niet op stap {k}, s = {s_k:.6f} m")
            print(message)

        theta3[k] = x[0]
        theta4[k] = x[1]
        theta5[k] = x[2]
        theta6[k] = x[3]
        theta7[k] = x[4]
        theta8[k] = x[5]

        residual_pos[k] = np.linalg.norm(
            loop_closure_eqs(
                x, s_k,
                r3a, r3b, r4a, r4b, r5a, r5b, r6, r7a, r7b, r8a
            )
        )

        # Snelheidsanalyse
        A = np.array([
            [-(r3a + r3b) * np.sin(theta3[k]),  r4a * np.sin(theta4[k]), 0, 0, 0, 0],
            [ (r3a + r3b) * np.cos(theta3[k]), -r4a * np.cos(theta4[k]), 0, 0, 0, 0],

            [ r3b * np.sin(theta3[k]),          r4b * np.sin(theta4[k]), -(r5a + r5b) * np.sin(theta5[k]), 0, -r7a * np.sin(theta7[k]), 0],
            [-r3b * np.cos(theta3[k]),         -r4b * np.cos(theta4[k]),  (r5a + r5b) * np.cos(theta5[k]), 0,  r7a * np.cos(theta7[k]), 0],

            [0, 0,  r5b * np.sin(theta5[k]), -r6 * np.sin(theta6[k]),  (r7a + r7b) * np.sin(theta7[k]), -r8a * np.sin(theta8[k])],
            [0, 0, -r5b * np.cos(theta5[k]),  r6 * np.cos(theta6[k]), -(r7a + r7b) * np.cos(theta7[k]),  r8a * np.cos(theta8[k])]
        ])

        B = np.array([
            0,
            ds[k],
            0,
            0,
            0,
            0
        ])

        x = np.linalg.solve(A, B)

        dtheta3[k] = x[0]
        dtheta4[k] = x[1]
        dtheta5[k] = x[2]
        dtheta6[k] = x[3]
        dtheta7[k] = x[4]
        dtheta8[k] = x[5]

        cond[k] = np.linalg.cond(A)

        # Versnellingsanalyse
        B = np.array([
            (r3a + r3b) * np.cos(theta3[k]) * dtheta3[k]**2 - r4a * np.cos(theta4[k]) * dtheta4[k]**2,

            dds[k]
            + (r3a + r3b) * np.sin(theta3[k]) * dtheta3[k]**2
            - r4a * np.sin(theta4[k]) * dtheta4[k]**2,

            -r3b * np.cos(theta3[k]) * dtheta3[k]**2
            + (r5a + r5b) * np.cos(theta5[k]) * dtheta5[k]**2
            + r7a * np.cos(theta7[k]) * dtheta7[k]**2
            - r4b * np.cos(theta4[k]) * dtheta4[k]**2,

            -r3b * np.sin(theta3[k]) * dtheta3[k]**2
            + (r5a + r5b) * np.sin(theta5[k]) * dtheta5[k]**2
            + r7a * np.sin(theta7[k]) * dtheta7[k]**2
            - r4b * np.sin(theta4[k]) * dtheta4[k]**2,

            -r5b * np.cos(theta5[k]) * dtheta5[k]**2
            + r6 * np.cos(theta6[k]) * dtheta6[k]**2
            + r8a * np.cos(theta8[k]) * dtheta8[k]**2
            - (r7a + r7b) * np.cos(theta7[k]) * dtheta7[k]**2,

            -r5b * np.sin(theta5[k]) * dtheta5[k]**2
            + r6 * np.sin(theta6[k]) * dtheta6[k]**2
            + r8a * np.sin(theta8[k]) * dtheta8[k]**2
            - (r7a + r7b) * np.sin(theta7[k]) * dtheta7[k]**2
        ])

        x = np.linalg.solve(A, B)

        ddtheta3[k] = x[0]
        ddtheta4[k] = x[1]
        ddtheta5[k] = x[2]
        ddtheta6[k] = x[3]
        ddtheta7[k] = x[4]
        ddtheta8[k] = x[5]

        # Branch tracking via vorige oplossing
        theta3_init = theta3[k] + dt * dtheta3[k]
        theta4_init = theta4[k] + dt * dtheta4[k]
        theta5_init = theta5[k] + dt * dtheta5[k]
        theta6_init = theta6[k] + dt * dtheta6[k]
        theta7_init = theta7[k] + dt * dtheta7[k]
        theta8_init = theta8[k] + dt * dtheta8[k]

    return theta3, theta4, theta5, theta6, theta7, theta8, \
           dtheta3, dtheta4, dtheta5, dtheta6, dtheta7, dtheta8, \
           ddtheta3, ddtheta4, ddtheta5, ddtheta6, ddtheta7, ddtheta8, \
           cond, residual_pos


# Initialisatie van opslagarrays
theta3   = np.zeros_like(t)
theta4   = np.zeros_like(t)
theta5   = np.zeros_like(t)
theta6   = np.zeros_like(t)
theta7   = np.zeros_like(t)
theta8   = np.zeros_like(t)

dtheta3  = np.zeros_like(t)
dtheta4  = np.zeros_like(t)
dtheta5  = np.zeros_like(t)
dtheta6  = np.zeros_like(t)
dtheta7  = np.zeros_like(t)
dtheta8  = np.zeros_like(t)

ddtheta3 = np.zeros_like(t)
ddtheta4 = np.zeros_like(t)
ddtheta5 = np.zeros_like(t)
ddtheta6 = np.zeros_like(t)
ddtheta7 = np.zeros_like(t)
ddtheta8 = np.zeros_like(t)

cond = np.zeros_like(t)
residual_pos = np.zeros_like(t)


# Uitvoeren van de volledige kinematische analyse
[theta3, theta4, theta5, theta6, theta7, theta8,
 dtheta3, dtheta4, dtheta5, dtheta6, dtheta7, dtheta8,
 ddtheta3, ddtheta4, ddtheta5, ddtheta6, ddtheta7, ddtheta8,
 cond, residual_pos] = kinematics_umbrella(
    r3a, r3b, r4a, r4b, r5a, r5b, r6, r7a, r7b, r8a,
    s, ds, dds,
    theta3_init, theta4_init, theta5_init, theta6_init, theta7_init, theta8_init,
    t
)

In [ ]:
# Bereken de volledige kinematica van het mechanisme

t_size = len(t)
sim_fraction = 2
frames = t_size / sim_fraction
delta = int(np.floor(t_size / frames))
index_vec = np.arange(0, t_size, delta)

theta3   = np.zeros_like(t)
theta4   = np.zeros_like(t)
theta5   = np.zeros_like(t)
theta6   = np.zeros_like(t)
theta7   = np.zeros_like(t)
theta8   = np.zeros_like(t)

dtheta3  = np.zeros_like(t)
dtheta4  = np.zeros_like(t)
dtheta5  = np.zeros_like(t)
dtheta6  = np.zeros_like(t)
dtheta7  = np.zeros_like(t)
dtheta8  = np.zeros_like(t)

ddtheta3 = np.zeros_like(t)
ddtheta4 = np.zeros_like(t)
ddtheta5 = np.zeros_like(t)
ddtheta6 = np.zeros_like(t)
ddtheta7 = np.zeros_like(t)
ddtheta8 = np.zeros_like(t)

cond = np.zeros_like(t)
residual_pos = np.zeros_like(t)

[theta3, theta4, theta5, theta6, theta7, theta8,
 dtheta3, dtheta4, dtheta5, dtheta6, dtheta7, dtheta8,
 ddtheta3, ddtheta4, ddtheta5, ddtheta6, ddtheta7, ddtheta8,
 cond, residual_pos] = kinematics_umbrella(
    r3a, r3b, r4a, r4b, r5a, r5b, r6, r7a, r7b, r8a,
    s, ds, dds,
    theta3_init, theta4_init, theta5_init, theta6_init, theta7_init, theta8_init,
    t
)

# Bepaal de outputpositie van punt K
Kx = np.zeros_like(t)
Ky = np.zeros_like(t)

for k in range(len(t)):
    C = np.array([0.0, 0.0])
    B = C + np.array([0.0, -s[k]])
    D = B + rotate_vector(np.array([r3a, 0.0]), theta3[k])
    F = D + rotate_vector(np.array([r5a, 0.0]), theta5[k])
    I = F + rotate_vector(np.array([r6, 0.0]), theta6[k])
    J = I + rotate_vector(np.array([r8a, 0.0]), theta8[k])
    K = J + rotate_vector(np.array([r8b, 0.0]), theta8[k])

    Kx[k] = K[0]
    Ky[k] = K[1]

# Positieplots
fig1, ax1 = plt.subplots(nrows=2, ncols=2, constrained_layout=True, figsize=(10, 8))
fig1.suptitle("Controle van de positie-oplossing")

ax1[0,0].plot(s, theta3, label=r'$\theta_3$')
ax1[0,0].plot(s, theta4, label=r'$\theta_4$')
ax1[0,0].plot(s, theta5, label=r'$\theta_5$')
ax1[0,0].plot(s, theta6, label=r'$\theta_6$')
ax1[0,0].plot(s, theta7, label=r'$\theta_7$')
ax1[0,0].plot(s, theta8, label=r'$\theta_8$')
ax1[0,0].set_xlabel(r'$s$ [m]')
ax1[0,0].set_ylabel(r'$\theta_i$ [rad]')
ax1[0,0].legend()

ax1[0,1].plot(s, Kx, label=r'$K_x$')
ax1[0,1].plot(s, Ky, label=r'$K_y$')
ax1[0,1].set_xlabel(r'$s$ [m]')
ax1[0,1].set_ylabel(r'$K_x, K_y$ [m]')
ax1[0,1].legend()

ax1[1,0].plot(Kx, Ky)
ax1[1,0].set_xlabel(r'$K_x$ [m]')
ax1[1,0].set_ylabel(r'$K_y$ [m]')
ax1[1,0].set_title("Traject van punt K")
ax1[1,0].axis('equal')

ax1[1,1].plot(s, residual_pos)
ax1[1,1].set_xlabel(r'$s$ [m]')
ax1[1,1].set_ylabel("closure residual [-]")
ax1[1,1].set_title("Numerieke sluitingsfout")

plt.show()

print(f"Maximale closure residual: {np.max(residual_pos):.3e}")
print(f"Gemiddelde closure residual: {np.mean(residual_pos):.3e}")

## Positievalidatie via dubbele kettingberekening

Voor gewrichten E, G en J zijn twee onafhankelijke kinematische ketens beschikbaar.
De absolute en relatieve afwijking tussen beide ketens kwantificeert de nauwkeurigheid
van de positie-oplossing, los van de sluitingsresiduals.

| Gewricht | Keten 1 | Keten 2 |
|----------|---------|---------|
| E | B → link 3 (r3a + r3b) | C → link 4 (r4a) |
| G | D → link 5 (r5a + r5b) | H → link 7 (r7a) |
| J | I → link 8 (r8a) | H → link 7 (r7a + r7b) |

In [ ]:
# Multi-gewricht positievalidatie via dubbele ketens

E3_pos = np.zeros((len(t), 2))
E4_pos = np.zeros((len(t), 2))
G5_pos = np.zeros((len(t), 2))
G7_pos = np.zeros((len(t), 2))
J8_pos = np.zeros((len(t), 2))
J7_pos = np.zeros((len(t), 2))

for k in range(len(t)):
    C    = np.array([0.0, 0.0])
    B    = C + np.array([0.0, -s[k]])
    D    = B    + rotate_vector(np.array([r3a, 0.0]),        theta3[k])
    F_pt = D    + rotate_vector(np.array([r5a, 0.0]),        theta5[k])
    I_pt = F_pt + rotate_vector(np.array([r6,  0.0]),        theta6[k])
    H    = C    + rotate_vector(np.array([r4a + r4b, 0.0]),  theta4[k])

    E3_pos[k] = D    + rotate_vector(np.array([r3b,       0.0]), theta3[k])
    E4_pos[k] = C    + rotate_vector(np.array([r4a,       0.0]), theta4[k])
    G5_pos[k] = F_pt + rotate_vector(np.array([r5b,       0.0]), theta5[k])
    G7_pos[k] = H    + rotate_vector(np.array([-r7a,      0.0]), theta7[k])
    J8_pos[k] = I_pt + rotate_vector(np.array([r8a,       0.0]), theta8[k])
    J7_pos[k] = H    + rotate_vector(np.array([r7b,       0.0]), theta7[k])

fig_val, axes = plt.subplots(3, 2, figsize=(14, 9), constrained_layout=True)
fig_val.suptitle("Positievalidatie: absolute en relatieve fout per gewricht")

joints_val = [
    ("E", E3_pos, E4_pos),
    ("G", G5_pos, G7_pos),
    ("J", J8_pos, J7_pos),
]

for i, (name, pos1, pos2) in enumerate(joints_val):
    abs_err = np.abs(pos1 - pos2)
    denom   = np.where(np.abs(pos1) > 1e-12, np.abs(pos1), 1e-12)
    rel_err = np.abs(abs_err / denom)

    axes[i, 0].plot(t, abs_err[:, 0], label=f'{name}_x')
    axes[i, 0].plot(t, abs_err[:, 1], label=f'{name}_y')
    axes[i, 0].set_yscale("log"); axes[i, 0].grid(True); axes[i, 0].legend()
    axes[i, 0].set_ylabel("Abs. fout [m]")
    axes[i, 0].set_title(f"Gewricht {name} -- absolute fout")

    axes[i, 1].plot(t, rel_err[:, 0], label=f'{name}_x')
    axes[i, 1].plot(t, rel_err[:, 1], label=f'{name}_y')
    axes[i, 1].set_yscale("log"); axes[i, 1].grid(True); axes[i, 1].legend()
    axes[i, 1].set_ylabel("Rel. fout [-]")
    axes[i, 1].set_title(f"Gewricht {name} -- relatieve fout")

for ax in axes[-1, :]:
    ax.set_xlabel("t [s]")

plt.show()

print("Max absolute fouten per gewricht:")
for name, pos1, pos2 in joints_val:
    err = np.max(np.abs(pos1 - pos2))
    print(f"  Gewricht {name}: {err:.3e} m")

## Animatie

De animatie reconstrueert voor opeenvolgende tijdstappen de volledige configuratie van het mechanisme. De vaste mast, de schuiver, de scharnierpunten en de bewegende links worden getekend op basis van dezelfde berekende hoeken als in de positie-analyse. Daardoor is de animatie geen aparte simulatie, maar een geometrische visualisatie van de numerieke oplossing.

Deze stap is nuttig omdat sommige fouten in een stangenmechanisme moeilijk uit grafieken alleen te herkennen zijn. Een verkeerde configuratietak, een onverwachte sprong van de solver of een inconsistent punt kan visueel sneller opvallen. De punten `E`, `G` en `J` worden in de animatie gemiddeld uit de twee constructies die uit verschillende lussen volgen. Als de sluitingsvergelijkingen goed voldaan zijn, vallen die constructies vrijwel samen en blijft het mechanisme gesloten.

De grenzen van het tekenvenster worden afgeleid uit `L1`, zodat de animatie bij redelijke parameterwijzigingen leesbaar blijft. Na een wijziging in de parametercel wordt dezelfde animatie opnieuw opgebouwd met de nieuwe geometrie en het nieuwe bewegingsverloop.

De animatie gebruikt dezelfde puntnamen als de analytische beschrijving. Daardoor blijft de relatie tussen de sluitingsvergelijkingen en de geometrische beweging duidelijk: punt `C` is vast, punt `B` volgt de schuiver, en punt `K` beschrijft de relevante outputbeweging van de buitenste rib.


In [ ]:
# instellingen voor de animatie

plt.ioff()
fig2, ax = plt.subplots()

t_size = len(t)
sim_fraction = 2                     # toon ongeveer de helft van de tijdstappen
frames = t_size / sim_fraction
delta = int(np.floor(t_size / frames))
index_vec = np.arange(0, t_size, delta)

# tekenvenster gebaseerd op de mastlengte
x_left   = -0.10 * L1
x_right  =  1.25 * L1
y_bottom = -1.05 * L1
y_top    =  0.75 * L1

In [ ]:
# animatie van het paraplu-mechanisme

def update(frame):
    global fig2, ax, index_vec

    k = index_vec[frame]

    ax.cla()
    ax.set_xlabel('[m]')
    ax.set_ylabel('[m]')
    ax.set_xlim([x_left, x_right])
    ax.set_ylim([y_bottom, y_top])
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'Tijd t = {t[k]:.2f} s')

    # vaste punt
    C = np.array([0.0, 0.0])

    # schuiverpunt
    B = C + np.array([0.0, -s[k]])

    # link 3: B - D - E
    D = B + rotate_vector(np.array([r3a, 0.0]), theta3[k])
    E_from_3 = D + rotate_vector(np.array([r3b, 0.0]), theta3[k])

    # link 4: C - E - H
    E_from_4 = C + rotate_vector(np.array([r4a, 0.0]), theta4[k])
    H = E_from_4 + rotate_vector(np.array([r4b, 0.0]), theta4[k])

    # gemiddeld punt E voor visuele sluiting
    E = 0.5 * (E_from_3 + E_from_4)

    # link 5: D - F - G
    F = D + rotate_vector(np.array([r5a, 0.0]), theta5[k])
    G_from_5 = F + rotate_vector(np.array([r5b, 0.0]), theta5[k])

    # link 6: F - I
    I = F + rotate_vector(np.array([r6, 0.0]), theta6[k])

    # link 7: H - G en H - J
    J_from_7 = H + rotate_vector(np.array([r7b, 0.0]), theta7[k])
    G_from_7 = H - rotate_vector(np.array([r7a, 0.0]), theta7[k])

    # gemiddeld punt G voor visuele sluiting
    G = 0.5 * (G_from_5 + G_from_7)

    # link 8: I - J - K
    J_from_8 = I + rotate_vector(np.array([r8a, 0.0]), theta8[k])
    J = 0.5 * (J_from_7 + J_from_8)
    K = J + rotate_vector(np.array([r8b, 0.0]), theta8[k])

    # mast
    mast = np.array([[0.0, 0.0], [0.0, -L1]])
    ax.plot(mast[:, 0], mast[:, 1], 'k-', linewidth=3)

    # links
    ax.plot([B[0], D[0], E[0]], [B[1], D[1], E[1]], '-o', linewidth=3)   # link 3
    ax.plot([C[0], E[0], H[0]], [C[1], E[1], H[1]], '-o', linewidth=3)   # link 4
    ax.plot([D[0], F[0], G[0]], [D[1], F[1], G[1]], '-o', linewidth=3)   # link 5
    ax.plot([F[0], I[0]],       [F[1], I[1]],       '-o', linewidth=3)   # link 6
    ax.plot([G[0], H[0], J[0]], [G[1], H[1], J[1]], '-o', linewidth=3)   # link 7
    ax.plot([I[0], J[0], K[0]], [I[1], J[1], K[1]], '-o', linewidth=3)   # link 8

    # schuiver
    ax.plot(B[0], B[1], 'rs', markersize=8)

    # labels
    labels = {
        "C": C, "B": B, "D": D, "E": E, "F": F,
        "G": G, "H": H, "I": I, "J": J, "K": K
    }

    for name, P in labels.items():
        ax.text(P[0] + 0.01, P[1] + 0.01, name, fontsize=10)

    return ax


ani = FuncAnimation(fig2, update, frames=len(index_vec), interval=50, repeat=False)

ani_html = ani.to_jshtml(default_mode='once')
plt.close(fig2)  # sluit fig2 zodat de widget niet verschijnt in latere figuren
HTML(ani_html)

## Snelheidsanalyse

De snelheidsanalyse ontstaat door de positie-sluitingsvergelijkingen naar de tijd af te leiden. Omdat de linklengtes constant zijn, komen in deze afgeleide vergelijkingen de hoeksnelheden van de links en de snelheid van de schuiver voor. De onbekenden zijn de zes hoeksnelheden `dtheta3` tot en met `dtheta8`. De gekende invoer is `ds`, de tijdsafgeleide van de schuiverpositie.

De afgeleide vergelijkingen kunnen voor elke configuratie geschreven worden als een lineair stelsel `A x = B`. De matrix `A` hangt af van de actuele hoeken en geometrische parameters. Ze beschrijft hoe veranderingen in de linkhoeken samenhangen met de sluitingsvoorwaarden. Het rechterlid `B` bevat de bijdrage van de opgelegde schuiversnelheid. In de code worden de hoeksnelheden al tijdens de kinematische berekening bepaald; de extra controlecel bouwt `A` en `B` expliciet opnieuw op en controleert het residu van het snelheidsstelsel.

Naast de interne hoeksnelheden wordt de snelheid van punt `K` berekend. Hiervoor wordt de snelheid opgebouwd langs de keten van punten tot aan de buitenste rib. De componenten `Kdot_x` en `Kdot_y` geven de richting van de beweging, terwijl `Kdot_norm` de grootte van de puntsnelheid samenvat. De figuren als functie van tijd en schuiverpositie maken zichtbaar waar de beweging sneller of trager verloopt.

De snelheidsanalyse vormt ook de brug naar de conditioneringsanalyse, omdat dezelfde matrix `A` gebruikt wordt om te beoordelen hoe gevoelig het mechanisme in een bepaalde configuratie is.

De matrix `A` is dezelfde structuur die later gebruikt wordt om de conditionering te beoordelen. Een correcte snelheidsoplossing vereist dus niet alleen een klein residu, maar ook een matrix die voldoende goed geconditioneerd blijft. Bij slechte conditionering kunnen relatief kleine wijzigingen in `ds` of in de geometrie grote verschillen in de berekende hoeksnelheden veroorzaken.


In [ ]:
def compute_A_matrix(theta, r3a, r3b, r4a, r4b, r5a, r5b, r6, r7a, r7b, r8a):
    theta3, theta4, theta5, theta6, theta7, theta8 = theta

    A = np.array([
        [-(r3a + r3b) * np.sin(theta3),  r4a * np.sin(theta4), 0, 0, 0, 0],
        [ (r3a + r3b) * np.cos(theta3), -r4a * np.cos(theta4), 0, 0, 0, 0],

        [ r3b * np.sin(theta3),          r4b * np.sin(theta4), -(r5a + r5b) * np.sin(theta5), 0, -r7a * np.sin(theta7), 0],
        [-r3b * np.cos(theta3),         -r4b * np.cos(theta4),  (r5a + r5b) * np.cos(theta5), 0,  r7a * np.cos(theta7), 0],

        [0, 0,  r5b * np.sin(theta5), -r6 * np.sin(theta6),  (r7a + r7b) * np.sin(theta7), -r8a * np.sin(theta8)],
        [0, 0, -r5b * np.cos(theta5),  r6 * np.cos(theta6), -(r7a + r7b) * np.cos(theta7),  r8a * np.cos(theta8)]
    ])

    return A

In [ ]:
# Snelheidsanalyse op basis van de reeds uitgevoerde kinematische analyse
# De hoeksnelheden dtheta_i zijn al berekend in kinematics_umbrella().
# In deze cel bouwen we expliciet A en B op, controleren we het snelheidsstelsel
# en bepalen we de relevante outputsnelheden van punt K.

def perp(v):
    """2D-operator voor omega x r = [-omega*y, omega*x]."""
    return np.array([-v[1], v[0]])

# opslag
vel_residual = np.zeros_like(t)
Kdot_x = np.zeros_like(t)
Kdot_y = np.zeros_like(t)
Kdot_norm = np.zeros_like(t)

for k in range(len(t)):
    theta_k = np.array([
        theta3[k],
        theta4[k],
        theta5[k],
        theta6[k],
        theta7[k],
        theta8[k]
    ])

    omega_k = np.array([
        dtheta3[k],
        dtheta4[k],
        dtheta5[k],
        dtheta6[k],
        dtheta7[k],
        dtheta8[k]
    ])

    # matrix A van het snelheidsstelsel
    A = compute_A_matrix(theta_k, r3a, r3b, r4a, r4b, r5a, r5b, r6, r7a, r7b, r8a)

    # rechterlid B
    B = np.array([
        0.0,
        ds[k],
        0.0,
        0.0,
        0.0,
        0.0
    ])

    # controle van het snelheidsstelsel A * omega = B
    vel_residual[k] = np.linalg.norm(A @ omega_k - B)

    # --- relevante puntsnelheden bepalen ---
    # Posities van de nodige punten
    C = np.array([0.0, 0.0])
    Bp = C + np.array([0.0, -s[k]])

    D = Bp + rotate_vector(np.array([r3a, 0.0]), theta3[k])
    F = D  + rotate_vector(np.array([r5a, 0.0]), theta5[k])
    I = F  + rotate_vector(np.array([r6, 0.0]),  theta6[k])
    J = I  + rotate_vector(np.array([r8a, 0.0]), theta8[k])
    K = J  + rotate_vector(np.array([r8b, 0.0]), theta8[k])

    # snelheden van opeenvolgende punten
    vB = np.array([0.0, -ds[k]])

    r_BD = D - Bp
    vD = vB + dtheta3[k] * perp(r_BD)

    r_DF = F - D
    vF = vD + dtheta5[k] * perp(r_DF)

    r_FI = I - F
    vI = vF + dtheta6[k] * perp(r_FI)

    r_IK = K - I
    vK = vI + dtheta8[k] * perp(r_IK)

    Kdot_x[k] = vK[0]
    Kdot_y[k] = vK[1]
    Kdot_norm[k] = np.linalg.norm(vK)

print("Maximale residual van het snelheidsstelsel:")
print(f"max ||A·omega - B|| = {np.max(vel_residual):.3e}")

print("\nMaximale norm van de snelheid van punt K:")
print(f"max |v_K| = {np.max(Kdot_norm):.6f} m/s")

In [ ]:
# Snelheidsanalyse: secundaire output en punt K
# Kdot_x, Kdot_y, Kdot_norm zijn berekend in cel orig-018 via kettingpropagatie.

# Grafieken als functie van de tijd
fig_vel_t, ax_vel_t = plt.subplots(nrows=2, ncols=2, constrained_layout=True, figsize=(12, 7))
fig_vel_t.suptitle("Snelheidsanalyse van link 8 en punt K als functie van de tijd")

ax_vel_t[0,0].plot(t, dtheta8)
ax_vel_t[0,0].set_ylabel(r'$\dot{\theta}_8$ [rad/s]')
ax_vel_t[0,0].set_xlabel('t [s]')
ax_vel_t[0,0].set_title(r'Hoeksnelheid van link 8')

ax_vel_t[0,1].plot(t, Kdot_x)
ax_vel_t[0,1].set_ylabel(r'$\dot{K}_x$ [m/s]')
ax_vel_t[0,1].set_xlabel('t [s]')
ax_vel_t[0,1].set_title(r'Snelheid van K in x-richting')

ax_vel_t[1,0].plot(t, Kdot_y)
ax_vel_t[1,0].set_ylabel(r'$\dot{K}_y$ [m/s]')
ax_vel_t[1,0].set_xlabel('t [s]')
ax_vel_t[1,0].set_title(r'Snelheid van K in y-richting')

ax_vel_t[1,1].plot(t, Kdot_norm)
ax_vel_t[1,1].set_ylabel(r'$||\dot{K}||$ [m/s]')
ax_vel_t[1,1].set_xlabel('t [s]')
ax_vel_t[1,1].set_title(r'Norm van de snelheid van K')

plt.show()


# Grafieken als functie van de schuiverpositie
fig_vel_s, ax_vel_s = plt.subplots(nrows=2, ncols=2, constrained_layout=True, figsize=(12, 7))
fig_vel_s.suptitle("Snelheidsanalyse van link 8 en punt K als functie van de schuiverpositie")

ax_vel_s[0,0].plot(s, dtheta8)
ax_vel_s[0,0].set_ylabel(r'$\dot{\theta}_8$ [rad/s]')
ax_vel_s[0,0].set_xlabel('s [m]')
ax_vel_s[0,0].set_title(r'Hoeksnelheid van link 8')

ax_vel_s[0,1].plot(s, Kdot_x)
ax_vel_s[0,1].set_ylabel(r'$\dot{K}_x$ [m/s]')
ax_vel_s[0,1].set_xlabel('s [m]')
ax_vel_s[0,1].set_title(r'Snelheid van K in x-richting')

ax_vel_s[1,0].plot(s, Kdot_y)
ax_vel_s[1,0].set_ylabel(r'$\dot{K}_y$ [m/s]')
ax_vel_s[1,0].set_xlabel('s [m]')
ax_vel_s[1,0].set_title(r'Snelheid van K in y-richting')

ax_vel_s[1,1].plot(s, Kdot_norm)
ax_vel_s[1,1].set_ylabel(r'$||\dot{K}||$ [m/s]')
ax_vel_s[1,1].set_xlabel('s [m]')
ax_vel_s[1,1].set_title(r'Norm van de snelheid van K')

plt.show()


# Hoeksnelheden van alle links
fig_omega, axes_omega = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
fig_omega.suptitle("Hoeksnelheden van alle links (als functie van de tijd)")

dtheta_all = [dtheta3, dtheta4, dtheta5, dtheta6, dtheta7, dtheta8]
labels_omega = [r'$\omega_3$', r'$\omega_4$', r'$\omega_5$',
                r'$\omega_6$', r'$\omega_7$', r'$\omega_8$']

for ax, dth, lbl in zip(axes_omega.flat, dtheta_all, labels_omega):
    ax.plot(t, dth, label=lbl)
    ax.set_ylabel(f'{lbl} [rad/s]')
    ax.set_xlabel('t [s]')
    ax.grid(True); ax.legend()

plt.show()

## Versnellingsanalyse

De versnellingsanalyse volgt uit de tijdsafgeleide van het snelheidsstelsel. Naast de hoekversnellingen ontstaan daarbij termen die afhangen van het kwadraat van de hoeksnelheden. Die termen komen voort uit de afgeleiden van sinus- en cosinustermen in de rotaties van de links. De onbekenden zijn de hoekversnellingen `ddtheta3` tot en met `ddtheta8`; de gekende invoer is de schuiverversnelling `dds`.

De notebook berekent de hoekversnellingen samen met de posities en snelheden in de kinematische routine. Daarna worden de afgeleide grootheden van punt `K` bepaald. De componenten `Kddot_x` en `Kddot_y` beschrijven hoe het outputpunt versnelt in het vaste assenstelsel. De norm `Kddot_norm` geeft een compacte maat voor de totale versnelling van het punt.

Versnellingen zijn belangrijk omdat ze rechtstreeks verband houden met inerti?le effecten. Ook wanneer deze notebook zelf geen krachten berekent, geven pieken in versnelling aan waar de dynamische belasting later groter kan worden. De grafieken van de hoekversnelling van link 8 en de versnelling van punt `K` tonen daarom welke delen van de cyclus kinematisch het meest belastend zijn.

De versnellingsanalyse gebruikt dezelfde positie- en snelheidsoplossingen als de vorige secties. Daardoor blijven alle grootheden gekoppeld aan ??n consistente beweging van het mechanisme.

De versnelling van een punt op een roterende link bevat zowel een tangentieel deel, evenredig met de hoekversnelling, als een normaal deel, evenredig met het kwadraat van de hoeksnelheid. Daarom kunnen versnellingen ook groot worden wanneer de hoekversnelling zelf niet extreem lijkt. Dit verklaart waarom de norm van de versnelling van punt `K` apart wordt bijgehouden.


In [ ]:
# Versnellingsanalyse: afgeleide grootheden van punt K
# De hoekversnellingen ddtheta3 ... ddtheta8 zijn reeds berekend in kinematics_umbrella().

Kddot_x = np.zeros_like(t)
Kddot_y = np.zeros_like(t)
Kddot_norm = np.zeros_like(t)

for k in range(len(t)):
    theta8_k = theta8[k]
    dtheta8_k = dtheta8[k]
    ddtheta8_k = ddtheta8[k]

    # vectoren op link 8
    r_IJ = rotate_vector(np.array([r8a, 0.0]), theta8_k)
    r_JK = rotate_vector(np.array([r8b, 0.0]), theta8_k)
    r_IK = r_IJ + r_JK

    x_IK = r_IK[0]
    y_IK = r_IK[1]

    # punt I
    C = np.array([0.0, 0.0])
    B = C + np.array([0.0, -s[k]])
    D = B + rotate_vector(np.array([r3a, 0.0]), theta3[k])
    F = D + rotate_vector(np.array([r5a, 0.0]), theta5[k])
    I = F + rotate_vector(np.array([r6, 0.0]), theta6[k])

    # snelheid van I
    vx_B = 0.0
    vy_B = -ds[k]

    vx_D = vx_B - r3a * np.sin(theta3[k]) * dtheta3[k]
    vy_D = vy_B + r3a * np.cos(theta3[k]) * dtheta3[k]

    vx_F = vx_D - r5a * np.sin(theta5[k]) * dtheta5[k]
    vy_F = vy_D + r5a * np.cos(theta5[k]) * dtheta5[k]

    vx_I = vx_F - r6 * np.sin(theta6[k]) * dtheta6[k]
    vy_I = vy_F + r6 * np.cos(theta6[k]) * dtheta6[k]

    # versnelling van I
    ax_B = 0.0
    ay_B = -dds[k]

    ax_D = ax_B - r3a * (
        np.cos(theta3[k]) * dtheta3[k]**2 + np.sin(theta3[k]) * ddtheta3[k]
    )
    ay_D = ay_B + r3a * (
        -np.sin(theta3[k]) * dtheta3[k]**2 + np.cos(theta3[k]) * ddtheta3[k]
    )

    ax_F = ax_D - r5a * (
        np.cos(theta5[k]) * dtheta5[k]**2 + np.sin(theta5[k]) * ddtheta5[k]
    )
    ay_F = ay_D + r5a * (
        -np.sin(theta5[k]) * dtheta5[k]**2 + np.cos(theta5[k]) * ddtheta5[k]
    )

    ax_I = ax_F - r6 * (
        np.cos(theta6[k]) * dtheta6[k]**2 + np.sin(theta6[k]) * ddtheta6[k]
    )
    ay_I = ay_F + r6 * (
        -np.sin(theta6[k]) * dtheta6[k]**2 + np.cos(theta6[k]) * ddtheta6[k]
    )

    # versnelling van K = versnelling van I + relatieve versnelling door rotatie van link 8
    Kddot_x[k] = ax_I - ddtheta8_k * y_IK - dtheta8_k**2 * x_IK
    Kddot_y[k] = ay_I + ddtheta8_k * x_IK - dtheta8_k**2 * y_IK

    Kddot_norm[k] = np.sqrt(Kddot_x[k]**2 + Kddot_y[k]**2)

In [ ]:
# Versnellingsanalyse: secundaire output en punt K
# Kddot_x, Kddot_y, Kddot_norm zijn berekend in cel orig-022 via kettingpropagatie.

# Grafieken als functie van de tijd
fig_acc_t, ax_acc_t = plt.subplots(nrows=2, ncols=2, constrained_layout=True, figsize=(12, 7))
fig_acc_t.suptitle("Versnellingsanalyse van link 8 en punt K als functie van de tijd")

ax_acc_t[0,0].plot(t, ddtheta8)
ax_acc_t[0,0].set_ylabel(r'$\ddot{\theta}_8$ [rad/s$^2$]')
ax_acc_t[0,0].set_xlabel('t [s]')
ax_acc_t[0,0].set_title(r'Hoekversnelling van link 8')

ax_acc_t[0,1].plot(t, Kddot_x)
ax_acc_t[0,1].set_ylabel(r'$\ddot{K}_x$ [m/s$^2$]')
ax_acc_t[0,1].set_xlabel('t [s]')
ax_acc_t[0,1].set_title(r'Versnelling van K in x-richting')

ax_acc_t[1,0].plot(t, Kddot_y)
ax_acc_t[1,0].set_ylabel(r'$\ddot{K}_y$ [m/s$^2$]')
ax_acc_t[1,0].set_xlabel('t [s]')
ax_acc_t[1,0].set_title(r'Versnelling van K in y-richting')

ax_acc_t[1,1].plot(t, Kddot_norm)
ax_acc_t[1,1].set_ylabel(r'$||\ddot{K}||$ [m/s$^2$]')
ax_acc_t[1,1].set_xlabel('t [s]')
ax_acc_t[1,1].set_title(r'Norm van de versnelling van K')

plt.show()


# Grafieken als functie van de schuiverpositie
fig_acc_s, ax_acc_s = plt.subplots(nrows=2, ncols=2, constrained_layout=True, figsize=(12, 7))
fig_acc_s.suptitle("Versnellingsanalyse van link 8 en punt K als functie van de schuiverpositie")

ax_acc_s[0,0].plot(s, ddtheta8)
ax_acc_s[0,0].set_ylabel(r'$\ddot{\theta}_8$ [rad/s$^2$]')
ax_acc_s[0,0].set_xlabel('s [m]')
ax_acc_s[0,0].set_title(r'Hoekversnelling van link 8')

ax_acc_s[0,1].plot(s, Kddot_x)
ax_acc_s[0,1].set_ylabel(r'$\ddot{K}_x$ [m/s$^2$]')
ax_acc_s[0,1].set_xlabel('s [m]')
ax_acc_s[0,1].set_title(r'Versnelling van K in x-richting')

ax_acc_s[1,0].plot(s, Kddot_y)
ax_acc_s[1,0].set_ylabel(r'$\ddot{K}_y$ [m/s$^2$]')
ax_acc_s[1,0].set_xlabel('s [m]')
ax_acc_s[1,0].set_title(r'Versnelling van K in y-richting')

ax_acc_s[1,1].plot(s, Kddot_norm)
ax_acc_s[1,1].set_ylabel(r'$||\ddot{K}||$ [m/s$^2$]')
ax_acc_s[1,1].set_xlabel('s [m]')
ax_acc_s[1,1].set_title(r'Norm van de versnelling van K')

plt.show()


# Hoekversnellingen van alle links
fig_alpha, axes_alpha = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
fig_alpha.suptitle("Hoekversnellingen van alle links (als functie van de tijd)")

ddtheta_all = [ddtheta3, ddtheta4, ddtheta5, ddtheta6, ddtheta7, ddtheta8]
labels_alpha = [r'$\alpha_3$', r'$\alpha_4$', r'$\alpha_5$',
                r'$\alpha_6$', r'$\alpha_7$', r'$\alpha_8$']

for ax, ddth, lbl in zip(axes_alpha.flat, ddtheta_all, labels_alpha):
    ax.plot(t, ddth, label=lbl, color='darkred')
    ax.set_ylabel(f'{lbl} [rad/s\u00b2]')
    ax.set_xlabel('t [s]')
    ax.grid(True); ax.legend()

plt.show()

## Conditionering en singulariteiten

De conditioneringsanalyse onderzoekt de matrix `A` uit het snelheidsstelsel. Voor elke configuratie wordt de condition number `cond(A)` berekend. Een lage condition number betekent dat het lineaire stelsel numeriek goed geconditioneerd is: kleine veranderingen in invoer of parameters leiden dan tot beperkte veranderingen in de berekende hoeksnelheden. Een hoge condition number wijst op een gevoeligere configuratie.

Wanneer `A` singulier wordt, kan het snelheidsstelsel geen unieke of stabiele oplossing meer leveren. Mechanisch komt dat overeen met een kritische configuratie, vaak omschreven als een singulariteit of dood punt. In zo'n stand kan de krachtoverdracht ongunstig worden en kunnen kleine verplaatsingen of meetfouten grote effecten hebben op de interne beweging. Het aantal vrijheidsgraden van het mechanisme verandert daardoor niet noodzakelijk, maar de numerieke en mechanische gevoeligheid neemt sterk toe.

De code bepaalt basisstatistieken van `cond(A)` en gebruikt een drempel om kritische zones te identificeren. Die drempel is het maximum van een absolute grens en een percentielgebaseerde grens. Daardoor blijft de analyse bruikbaar bij andere parameterwaarden, zonder dat de interpretatie volledig afhangt van ??n vaste schaal.

De twee figuren tonen `cond(A)` als functie van tijd en als functie van de schuiverpositie. De schuiverpositie is vaak de meest directe manier om een kritische configuratie aan de geometrie te koppelen.

Een hoge condition number is geen bewijs dat het mechanisme exact singulier is, maar wel een waarschuwing dat de configuratie dicht bij een ongunstige kinematische toestand kan liggen. De combinatie van de maximale waarde, de plaats van die waarde en het aantal kritische zones geeft daarom meer informatie dan ??n los getal.


In [ ]:
# Analyse van conditionering en kritische configuraties

import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. Basisstatistieken
# -----------------------------
cond_finite = cond[np.isfinite(cond)]

print("Statistiek van de condition number:")
print(f"minimum cond(A) = {np.min(cond_finite):.3e}")
print(f"gemiddelde cond(A) = {np.mean(cond_finite):.3e}")
print(f"maximum cond(A) = {np.max(cond_finite):.3e}")

# -----------------------------
# 2. Drempel voor kritische configuraties
# -----------------------------
cond_threshold_abs = 1e3
cond_threshold_percentile = np.percentile(cond_finite, 95)
cond_threshold = max(cond_threshold_abs, cond_threshold_percentile)

print(f"\nGebruikte drempel voor kritische configuraties: cond(A) > {cond_threshold:.3e}")

critical_idx = np.where(cond > cond_threshold)[0]

# -----------------------------
# 3. Kritische punten groeperen
# -----------------------------
critical_groups = []

if len(critical_idx) > 0:
    start = critical_idx[0]
    prev = critical_idx[0]

    for idx in critical_idx[1:]:
        if idx == prev + 1:
            prev = idx
        else:
            critical_groups.append((start, prev))
            start = idx
            prev = idx

    critical_groups.append((start, prev))

print(f"Aantal kritische zones: {len(critical_groups)}")

for i, (i_start, i_end) in enumerate(critical_groups, start=1):
    i_peak = i_start + np.argmax(cond[i_start:i_end+1])
    print(
        f"Zone {i}: "
        f"t ≈ {t[i_peak]:.3f} s, "
        f"s ≈ {s[i_peak]:.3f} m, "
        f"cond(A) ≈ {cond[i_peak]:.3e}"
    )

# Y-as begrenzen op de werkelijke datarange (+ 20% marge)
cond_max_data = np.max(cond_finite)
y_top = cond_max_data * 1.20
drempel_in_beeld = cond_threshold <= y_top

def _add_drempel(ax, cond_threshold, drempel_in_beeld):
    if drempel_in_beeld:
        ax.axhline(cond_threshold, linestyle='--', color='tab:orange',
                   label=f'kritische drempel ({cond_threshold:.0f})')
    else:
        ax.text(
            0.99, 0.97,
            f"drempel {cond_threshold:.0e} (buiten beeld)",
            transform=ax.transAxes,
            ha='right', va='top', fontsize=9,
            color='tab:orange',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='tab:orange', alpha=0.8)
        )

# -----------------------------
# 4. Plot cond(A) als functie van t
# -----------------------------
fig_cond_t, ax_cond_t = plt.subplots(figsize=(10, 4))
fig_cond_t.suptitle("Condition number van de snelheidsmatrix als functie van de tijd")

ax_cond_t.plot(t, cond, label='cond(A)')
_add_drempel(ax_cond_t, cond_threshold, drempel_in_beeld)

for (i_start, i_end) in critical_groups:
    i_peak = i_start + np.argmax(cond[i_start:i_end+1])
    ax_cond_t.plot(t[i_peak], cond[i_peak], 'o')
    ax_cond_t.annotate(
        f"s={s[i_peak]:.3f} m",
        (t[i_peak], cond[i_peak]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=9
    )

ax_cond_t.set_xlabel('t [s]')
ax_cond_t.set_ylabel('cond(A) [-]')
ax_cond_t.set_ylim(0, y_top)
ax_cond_t.legend()
plt.tight_layout()
plt.show()

# -----------------------------
# 5. Plot cond(A) als functie van s
# -----------------------------
fig_cond_s, ax_cond_s = plt.subplots(figsize=(10, 4))
fig_cond_s.suptitle("Condition number van de snelheidsmatrix als functie van de schuiverpositie")

ax_cond_s.plot(s, cond, label='cond(A)')
_add_drempel(ax_cond_s, cond_threshold, drempel_in_beeld)

for (i_start, i_end) in critical_groups:
    i_peak = i_start + np.argmax(cond[i_start:i_end+1])
    ax_cond_s.plot(s[i_peak], cond[i_peak], 'o')
    ax_cond_s.annotate(
        f"t={t[i_peak]:.2f} s",
        (s[i_peak], cond[i_peak]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=9
    )

ax_cond_s.set_xlabel('s [m]')
ax_cond_s.set_ylabel('cond(A) [-]')
ax_cond_s.set_ylim(0, y_top)
ax_cond_s.legend()
plt.tight_layout()
plt.show()


## Opslag voor Notebook 2

De dynamische analyse in Notebook 2 heeft de resultaten van deze kinematische analyse nodig. Daarom schrijft deze sectie de actuele waarden weg naar `notebook1_kinematica_results.npz`. Dat bestand bevat de tijdvector, het schuivertraject, de hoeken, hoeksnelheden, hoekversnellingen, de positie en afgeleiden van punt `K`, de conditionering, de sluitingsfout en alle geometrische parameters.

De `.npz` werkt als een tijdelijke overdracht tussen de kinematica en de dynamica. Wanneer bovenaan een parameter of het schuivertraject wordt aangepast, moeten alle cellen opnieuw uitgevoerd worden zodat dit bestand opnieuw overeenkomt met de actuele kinematische resultaten. Notebook 2 leest daarna dezelfde sleutelstructuur in en kan zijn dynamische berekeningen uitvoeren zonder de kinematische code te dupliceren.

De opslagcel print het absolute pad van het bestand. Dat maakt controleerbaar welke overdrachtsfile door de volgende notebook gebruikt wordt.

De opgeslagen grootheden zijn bewust numerieke arrays en scalars, geen figuren. Notebook 2 kan daardoor de resultaten opnieuw verwerken met eigen dynamische parameters, terwijl de kinematische basis ondubbelzinnig vastligt.


In [ ]:
# Opslaan van de relevante kinematische resultaten voor Notebook 2
from pathlib import Path

results_path = Path("notebook1_kinematica_results.npz").resolve()

np.savez(
    results_path,

    # tijd en invoer
    t=t,
    Ts=Ts,
    s=s,
    ds=ds,
    dds=dds,

    # configuratiehoeken
    theta3=theta3,
    theta4=theta4,
    theta5=theta5,
    theta6=theta6,
    theta7=theta7,
    theta8=theta8,

    # hoeksnelheden
    dtheta3=dtheta3,
    dtheta4=dtheta4,
    dtheta5=dtheta5,
    dtheta6=dtheta6,
    dtheta7=dtheta7,
    dtheta8=dtheta8,

    # hoekversnellingen
    ddtheta3=ddtheta3,
    ddtheta4=ddtheta4,
    ddtheta5=ddtheta5,
    ddtheta6=ddtheta6,
    ddtheta7=ddtheta7,
    ddtheta8=ddtheta8,

    # outputpunt K
    Kx=Kx,
    Ky=Ky,
    Kdot_x=Kdot_x,
    Kdot_y=Kdot_y,
    Kdot_norm=Kdot_norm,
    Kddot_x=Kddot_x,
    Kddot_y=Kddot_y,
    Kddot_norm=Kddot_norm,

    # numerieke diagnostiek
    cond=cond,
    residual_pos=residual_pos,

    # geometrische parameters
    L1=L1,
    r3a=r3a,
    r3b=r3b,
    r4a=r4a,
    r4b=r4b,
    r5a=r5a,
    r5b=r5b,
    r6=r6,
    r7a=r7a,
    r7b=r7b,
    r8a=r8a,
    r8b=r8b
)

print("Kinematische resultaten opgeslagen voor Notebook 2:")
print(results_path)


## Samenvatting

De laatste cel bundelt de belangrijkste numerieke kenmerken van de kinematische analyse. Ze geeft het gebruikte schuiverbereik, de maximale sluitingsfout, de maximale snelheid en versnelling van punt `K`, en de belangrijkste statistieken van `cond(A)`. Deze waarden zijn nuttig als compacte controle van een volledige run.

Bij een parameterwijziging veranderen deze getallen mee. De samenvatting maakt daardoor snel zichtbaar of de gewijzigde geometrie nog numeriek consistent is, of de outputbeweging sterker versnelt, en of de conditionering slechter of beter wordt. De cel vermeldt ook hoeveel kritische zones boven de gekozen conditioneringsdrempel gevonden werden. Samen met de figuren vormt dit een korte eindcontrole van de volledige kinematische berekening.

De samenvatting vervangt de figuren niet, maar vormt een numerieke eindcontrole. Vooral de sluitingsfout en de maximale conditionering zijn nuttig om te controleren of een volledige run betrouwbaar verlopen is.


In [ ]:
# Compacte samenvatting van de kinematische analyse
cond_finite_summary = cond[np.isfinite(cond)]
i_cond_max = int(np.nanargmax(cond))
i_speed_max = int(np.nanargmax(Kdot_norm))
i_acc_max = int(np.nanargmax(Kddot_norm))

print("SAMENVATTING - kinematica")
print("=" * 45)
print(f"Schuiverbereik s              : {np.min(s):.4f} m tot {np.max(s):.4f} m")
print(f"Maximale sluitingsfout        : {np.max(residual_pos):.3e}")
print(f"Maximale snelheid punt K      : {Kdot_norm[i_speed_max]:.4f} m/s bij s = {s[i_speed_max]:.4f} m")
print(f"Maximale versnelling punt K   : {Kddot_norm[i_acc_max]:.4f} m/s^2 bij s = {s[i_acc_max]:.4f} m")
print(f"cond(A) min / gem / max       : {np.min(cond_finite_summary):.3e} / {np.mean(cond_finite_summary):.3e} / {np.max(cond_finite_summary):.3e}")
print(f"Grootste cond(A)              : bij t = {t[i_cond_max]:.4f} s en s = {s[i_cond_max]:.4f} m")
print(f"Aantal kritische zones        : {len(critical_groups)}")

if len(critical_groups) == 0:
    print("Interpretatie singulariteiten : geen kritische zone boven de gekozen drempel in dit bewegingsbereik")
else:
    print("Kritische zones:")
    for zone_id, group in enumerate(critical_groups, start=1):
        group = np.asarray(group)
        i_peak = group[np.argmax(cond[group])]
        print(f"  zone {zone_id}: s = {s[i_peak]:.4f} m, t = {t[i_peak]:.4f} s, cond(A) = {cond[i_peak]:.3e}")

print("\nOverdrachtsbestand voor Notebook 2:")
print(results_path)
